In [1]:
import numpy as np

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),        # fixed size
    transforms.ToTensor(),                 # convert to tensor
    transforms.Normalize(                  # normalize
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [7]:
dataset = datasets.ImageFolder("105_classes_pins_dataset", transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [5]:
for i, (images, labels) in enumerate(loader):
    print(f"Batch {i}:", images.shape, labels.shape)
    if i == 2:
        break



Batch 0: torch.Size([32, 3, 224, 224]) torch.Size([32])
Batch 1: torch.Size([32, 3, 224, 224]) torch.Size([32])
Batch 2: torch.Size([32, 3, 224, 224]) torch.Size([32])


In [8]:
dataset_path = "105_classes_pins_dataset"

dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)


In [7]:
dataset.classes

['pins_Adriana Lima',
 'pins_Alex Lawther',
 'pins_Alexandra Daddario',
 'pins_Alvaro Morte',
 'pins_Amanda Crew',
 'pins_Andy Samberg',
 'pins_Anne Hathaway',
 'pins_Anthony Mackie',
 'pins_Avril Lavigne',
 'pins_Ben Affleck',
 'pins_Bill Gates',
 'pins_Bobby Morley',
 'pins_Brenton Thwaites',
 'pins_Brian J. Smith',
 'pins_Brie Larson',
 'pins_Chris Evans',
 'pins_Chris Hemsworth',
 'pins_Chris Pratt',
 'pins_Christian Bale',
 'pins_Cristiano Ronaldo',
 'pins_Danielle Panabaker',
 'pins_Dominic Purcell',
 'pins_Dwayne Johnson',
 'pins_Eliza Taylor',
 'pins_Elizabeth Lail',
 'pins_Emilia Clarke',
 'pins_Emma Stone',
 'pins_Emma Watson',
 'pins_Gwyneth Paltrow',
 'pins_Henry Cavil',
 'pins_Hugh Jackman',
 'pins_Inbar Lavi',
 'pins_Irina Shayk',
 'pins_Jake Mcdorman',
 'pins_Jason Momoa',
 'pins_Jennifer Lawrence',
 'pins_Jeremy Renner',
 'pins_Jessica Barden',
 'pins_Jimmy Fallon',
 'pins_Johnny Depp',
 'pins_Josh Radnor',
 'pins_Katharine Mcphee',
 'pins_Katherine Langford',
 'pins_Ke

In [8]:
print("Total images:", len(dataset))
print("Classes:", dataset.classes)
print("Number of classes:", len(dataset.classes))


Total images: 17534
Classes: ['pins_Adriana Lima', 'pins_Alex Lawther', 'pins_Alexandra Daddario', 'pins_Alvaro Morte', 'pins_Amanda Crew', 'pins_Andy Samberg', 'pins_Anne Hathaway', 'pins_Anthony Mackie', 'pins_Avril Lavigne', 'pins_Ben Affleck', 'pins_Bill Gates', 'pins_Bobby Morley', 'pins_Brenton Thwaites', 'pins_Brian J. Smith', 'pins_Brie Larson', 'pins_Chris Evans', 'pins_Chris Hemsworth', 'pins_Chris Pratt', 'pins_Christian Bale', 'pins_Cristiano Ronaldo', 'pins_Danielle Panabaker', 'pins_Dominic Purcell', 'pins_Dwayne Johnson', 'pins_Eliza Taylor', 'pins_Elizabeth Lail', 'pins_Emilia Clarke', 'pins_Emma Stone', 'pins_Emma Watson', 'pins_Gwyneth Paltrow', 'pins_Henry Cavil', 'pins_Hugh Jackman', 'pins_Inbar Lavi', 'pins_Irina Shayk', 'pins_Jake Mcdorman', 'pins_Jason Momoa', 'pins_Jennifer Lawrence', 'pins_Jeremy Renner', 'pins_Jessica Barden', 'pins_Jimmy Fallon', 'pins_Johnny Depp', 'pins_Josh Radnor', 'pins_Katharine Mcphee', 'pins_Katherine Langford', 'pins_Keanu Reeves', '

In [9]:
from torch.utils.data import random_split

train_size = int(0.7 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset, [train_size, val_size, test_size]
)


In [10]:


train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [12]:
import torch.nn as nn
model = nn.Sequential(
    nn.Conv2d(in_channels=3,out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 224 -> 112
    
    nn.Conv2d(in_channels=32,out_channels=64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 112 -> 56
    
    nn.Conv2d(in_channels=64,out_channels= 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 56 -> 28
    
    nn.Flatten(),
    nn.Linear(128*28*28, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 105)
)

In [13]:
# 6️⃣ Loss and Optimizer
import torch.optim as optim 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
# 7️⃣ Training Loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images, labels
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images, labels
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.4f}\n")

Epoch [1/10] Loss: 4.6477 Acc: 0.0139
Validation Accuracy: 0.0202

Epoch [2/10] Loss: 4.3296 Acc: 0.0465
Validation Accuracy: 0.0806



KeyboardInterrupt: 

In [ ]:

# 6️⃣ Testing (After Training)
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_acc = test_correct / test_total
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
print("\nFirst 10 Predictions:")
for i in range(10):
    print(
        f"True: {class_names[all_true[i]]} | "
        f"Predicted: {class_names[all_pred[i]]}"
    )
    

In [ ]:
torch.save(model.state_dict(), "face_recognition_resnet18.pth")
print("Model saved successfully")
